# Agregação do ENEM 2024 por Escola e por Item de CN

## 1. Configuração

In [ ]:
import polars as pl
from pathlib import Path

# Caminhos principais do projeto
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
RAW = ROOT / "data" / "raw"
INDICADORES = ROOT / "data" / "processed" / "indicadores"
ENEM = ROOT / "data" / "processed" / "enem"

ENEM.mkdir(parents=True, exist_ok=True)
itens_cn = pl.read_parquet(ENEM / "itens_cn_2024.parquet")

## 2. Máscara de Correção por Disciplina (Física, Química e Biologia)

In [ ]:
# Considera apenas itens válidos (IN_ITEM_ABAN == 0) mapeados em cada caderno CO_PROVA
validos = itens_cn.filter(pl.col("IN_FISICA").is_not_null() & (pl.col("IN_ITEM_ABAN") == 0))

linhas_mascara = []
for (co_prova,), grupo in validos.group_by(["CO_PROVA"]):
    f_idx = set(grupo.filter(pl.col("IN_FISICA") == 1)["IDX_POSICAO"].to_list())
    q_idx = set(grupo.filter(pl.col("IN_QUIMICA") == 1)["IDX_POSICAO"].to_list())
    b_idx = set(grupo.filter(pl.col("IN_BIOLOGIA") == 1)["IDX_POSICAO"].to_list())
    linha = {
        "CO_PROVA_CN": co_prova,
        "N_ITENS_FIS": len(f_idx),
        "N_ITENS_QUI": len(q_idx),
        "N_ITENS_BIO": len(b_idx),
    }
    for i in range(45):
        linha[f"F_{i}"] = 1 if i in f_idx else 0
        linha[f"Q_{i}"] = 1 if i in q_idx else 0
        linha[f"B_{i}"] = 1 if i in b_idx else 0
    linhas_mascara.append(linha)

df_mascara = pl.DataFrame(linhas_mascara)
print("Cadernos mapeados para correção:", df_mascara.height)

## 3. Leitura e Pontuação dos Alunos com Escola

In [ ]:
# Compara resposta e gabarito em cada uma das 45 posições de CN
acertos_pos = [
    (pl.col("TX_RESPOSTAS_CN").str.slice(i, 1) == pl.col("TX_GABARITO_CN").str.slice(i, 1))
    .cast(pl.Int8).alias(f"H_{i}")
    for i in range(45)
]

# Lê apenas candidatos vinculados a uma escola (CO_ESCOLA não nulo)
lf_alunos = (
    pl.scan_csv(
        RAW / "microdados_enem_2024" / "RESULTADOS_2024.csv",
        separator=";", encoding="utf8-lossy"
    )
    .filter(pl.col("CO_ESCOLA").is_not_null())
    .join(df_mascara.lazy(), on="CO_PROVA_CN", how="left")
    .with_columns(acertos_pos)
    .with_columns([
        pl.when(pl.col("N_ITENS_FIS").is_not_null() & (pl.col("TP_PRESENCA_CN") == 1))
        .then(pl.sum_horizontal([pl.col(f"H_{i}") * pl.col(f"F_{i}") for i in range(45)]))
        .alias("ACERTOS_FISICA"),
        pl.when(pl.col("N_ITENS_QUI").is_not_null() & (pl.col("TP_PRESENCA_CN") == 1))
        .then(pl.sum_horizontal([pl.col(f"H_{i}") * pl.col(f"Q_{i}") for i in range(45)]))
        .alias("ACERTOS_QUIMICA"),
        pl.when(pl.col("N_ITENS_BIO").is_not_null() & (pl.col("TP_PRESENCA_CN") == 1))
        .then(pl.sum_horizontal([pl.col(f"H_{i}") * pl.col(f"B_{i}") for i in range(45)]))
        .alias("ACERTOS_BIOLOGIA"),
    ])
    .with_columns([
        (pl.col("ACERTOS_FISICA") / pl.col("N_ITENS_FIS") * 100.0).alias("PCT_ACERTO_FISICA"),
        (pl.col("ACERTOS_QUIMICA") / pl.col("N_ITENS_QUI") * 100.0).alias("PCT_ACERTO_QUIMICA"),
        (pl.col("ACERTOS_BIOLOGIA") / pl.col("N_ITENS_BIO") * 100.0).alias("PCT_ACERTO_BIOLOGIA"),
        (
            (pl.col("NU_NOTA_CN") + pl.col("NU_NOTA_CH") + pl.col("NU_NOTA_LC")
             + pl.col("NU_NOTA_MT") + pl.col("NU_NOTA_REDACAO")) / 5.0
        ).alias("NOTA_GERAL"),
        ((pl.col("TP_PRESENCA_CH") == 1) & (pl.col("TP_PRESENCA_LC") == 1)).cast(pl.Int8).alias("PRESENTE_DIA1"),
        ((pl.col("TP_PRESENCA_CN") == 1) & (pl.col("TP_PRESENCA_MT") == 1)).cast(pl.Int8).alias("PRESENTE_DIA2"),
    ])
    .with_columns(
        ((pl.col("PRESENTE_DIA1") == 1) & (pl.col("PRESENTE_DIA2") == 1)).cast(pl.Int8).alias("PRESENTE_AMBOS")
    )
)

## 4. Agregação por Escola (enem_escolas_2024.parquet)

In [ ]:
# Dicionário limpo de nomes de municípios a partir do Censo Escolar
municipios = (
    pl.read_parquet(INDICADORES / "data_afd.parquet")
    .select(
        pl.col("CO_MUNICIPIO").cast(pl.Int64, strict=False).alias("CO_MUNICIPIO_ESC"),
        pl.col("NO_MUNICIPIO").alias("NO_MUNICIPIO_ESC"),
    )
    .unique(subset=["CO_MUNICIPIO_ESC"])
)

# Agrega métricas de participação, acertos em Física/Química/Biologia e notas TRI
enem_escolas = (
    lf_alunos.group_by("CO_ESCOLA")
    .agg([
        pl.col("SG_UF_ESC").first(),
        pl.col("CO_MUNICIPIO_ESC").cast(pl.Int64, strict=False).first(),
        pl.col("TP_DEPENDENCIA_ADM_ESC").cast(pl.Int8, strict=False).first(),
        pl.col("TP_LOCALIZACAO_ESC").cast(pl.Int8, strict=False).first(),
        pl.col("TP_SIT_FUNC_ESC").cast(pl.Int8, strict=False).first(),
        # Participação e abstenção
        pl.len().alias("QTD_INSCRITOS_ESCOLA"),
        pl.col("PRESENTE_DIA1").sum().alias("QTD_PRESENTES_DIA1"),
        pl.col("PRESENTE_DIA2").sum().alias("QTD_PRESENTES_DIA2"),
        pl.col("PRESENTE_AMBOS").sum().alias("QTD_PRESENTES_AMBOS"),
        ((1.0 - pl.col("PRESENTE_DIA2").mean()) * 100.0).round(2).alias("TX_ABSTENCAO_DIA2"),
        # Desempenho específico em Física, Química e Biologia
        pl.col("ACERTOS_FISICA").mean().round(2).alias("MEDIA_ACERTOS_FISICA"),
        pl.col("PCT_ACERTO_FISICA").mean().round(2).alias("PCT_ACERTO_FISICA"),
        pl.col("ACERTOS_FISICA").std().round(2).alias("STD_ACERTOS_FISICA"),
        pl.col("ACERTOS_QUIMICA").mean().round(2).alias("MEDIA_ACERTOS_QUIMICA"),
        pl.col("PCT_ACERTO_QUIMICA").mean().round(2).alias("PCT_ACERTO_QUIMICA"),
        pl.col("ACERTOS_QUIMICA").std().round(2).alias("STD_ACERTOS_QUIMICA"),
        pl.col("ACERTOS_BIOLOGIA").mean().round(2).alias("MEDIA_ACERTOS_BIOLOGIA"),
        pl.col("PCT_ACERTO_BIOLOGIA").mean().round(2).alias("PCT_ACERTO_BIOLOGIA"),
        pl.col("ACERTOS_BIOLOGIA").std().round(2).alias("STD_ACERTOS_BIOLOGIA"),
        # Notas TRI e Redação (médias, medianas e desvios-padrão)
        *[
            expr
            for area in ["CN", "MT", "CH", "LC", "REDACAO"]
            for expr in (
                pl.col(f"NU_NOTA_{area}").mean().round(2).alias(f"MEDIA_{area}"),
                pl.col(f"NU_NOTA_{area}").median().round(2).alias(f"MEDIANA_{area}"),
                pl.col(f"NU_NOTA_{area}").std().round(2).alias(f"STD_{area}"),
            )
        ],
        *[pl.col(f"NU_NOTA_COMP{c}").mean().round(2).alias(f"MEDIA_COMP{c}") for c in range(1, 6)],
        pl.col("NOTA_GERAL").mean().round(2).alias("MEDIA_GERAL"),
        pl.col("NOTA_GERAL").median().round(2).alias("MEDIANA_GERAL"),
        pl.col("NOTA_GERAL").std().round(2).alias("STD_GERAL"),
    ])
    .with_columns((pl.col("QTD_PRESENTES_AMBOS") >= 10).alias("IN_MIN_10_ALUNOS"))
    .collect()
    .join(municipios, on="CO_MUNICIPIO_ESC", how="left")
    .sort("MEDIA_GERAL", descending=True, nulls_last=True)
)

enem_escolas.write_parquet(ENEM / "enem_escolas_2024.parquet")
print("Escolas agregadas:", enem_escolas.shape)
enem_escolas.head(5)

## 5. Desempenho e Distratores por Questão (enem_itens_desempenho_2024.parquet)

In [ ]:
# Coleta respostas das 45 posições dos alunos presentes em CN com escola
df_cn = (
    pl.scan_csv(
        RAW / "microdados_enem_2024" / "RESULTADOS_2024.csv",
        separator=";", encoding="utf8-lossy"
    )
    .filter(pl.col("CO_ESCOLA").is_not_null() & (pl.col("TP_PRESENCA_CN") == 1))
    .select(
        "CO_PROVA_CN", "SG_UF_ESC", "TP_DEPENDENCIA_ADM_ESC",
        *[pl.col("TX_RESPOSTAS_CN").str.slice(i, 1).alias(f"R_{i}") for i in range(45)]
    )
    .collect()
)

# Conta marcações de cada alternativa (A-E, branco ".", dupla "*") por posição
contagens = pl.concat([
    df_cn.group_by(["CO_PROVA_CN", "SG_UF_ESC", "TP_DEPENDENCIA_ADM_ESC", f"R_{i}"])
    .len()
    .rename({"CO_PROVA_CN": "CO_PROVA", f"R_{i}": "RESPOSTA"})
    .with_columns(pl.lit(i + 91).cast(pl.Int64).alias("CO_POSICAO"))
    for i in range(45)
])

# Une com o catálogo de itens para agregar pelo CO_ITEM canônico
meta_cols = [
    "CO_ITEM", "DISCIPLINA", "IN_FISICA", "IN_QUIMICA", "IN_BIOLOGIA",
    "CO_HABILIDADE", "TX_GABARITO", "IN_ITEM_ABAN", "NU_PARAM_A", "NU_PARAM_B", "NU_PARAM_C",
]

desempenho_itens = (
    contagens.join(
        itens_cn.filter(pl.col("IN_FISICA").is_not_null()),
        on=["CO_PROVA", "CO_POSICAO"], how="inner"
    )
    .group_by(meta_cols + ["SG_UF_ESC", "TP_DEPENDENCIA_ADM_ESC"])
    .agg([
        pl.col("len").sum().alias("QTD_RESPOSTAS"),
        pl.col("len").filter(pl.col("RESPOSTA") == pl.col("TX_GABARITO")).sum().alias("QTD_ACERTOS"),
        *[pl.col("len").filter(pl.col("RESPOSTA") == alt).sum().alias(f"QTD_{alt}") for alt in ["A", "B", "C", "D", "E"]],
        pl.col("len").filter(pl.col("RESPOSTA") == ".").sum().alias("QTD_BRANCO"),
        pl.col("len").filter(pl.col("RESPOSTA") == "*").sum().alias("QTD_DUPLA"),
    ])
    .with_columns([
        (pl.col("QTD_ACERTOS") / pl.col("QTD_RESPOSTAS") * 100.0).round(2).alias("PCT_ACERTO"),
        *[(pl.col(f"QTD_{alt}") / pl.col("QTD_RESPOSTAS") * 100.0).round(2).alias(f"PCT_{alt}") for alt in ["A", "B", "C", "D", "E"]],
    ])
    .sort(["DISCIPLINA", "CO_ITEM", "SG_UF_ESC", "TP_DEPENDENCIA_ADM_ESC"])
)

desempenho_itens.write_parquet(ENEM / "enem_itens_desempenho_2024.parquet")
print("Estatísticas por item salvas:", desempenho_itens.shape)
desempenho_itens.filter(pl.col("IN_FISICA") == 1).head(5)